Objective

This notebook focuses on implementing and evaluating Supervised Learning algorithms to classify MRI scans as "Healthy" (0) or "Tumor" (1). The goal is to compare a linear baseline (Logistic Regression) against an ensemble method (Random Forest), utilizing 5-fold cross-validation and standardized evaluation metrics to determine the most reliable model for clinical diagnostic support.
Required Inputs

    File Path: ../data/cleaned.csv (The processed dataset generated in Task 1).

Outputs Produced

    Trained Model: ../models/supervised_best.pkl (The saved Random Forest Classifier).

    Evaluation Data: ../data/residuals.csv (A file containing test set predictions and error analysis for downstream tasks).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import os

df = pd.read_csv('../data/cleaned.csv')
print(f"Dataset Cleaned : {df.shape}")



Task Selection: Classification

Classification is the appropriate task because the target variable (Class) is categorical and binary. We are predicting whether an MRI scan belongs to one of two distinct groups: 0 (Healthy) or 1 (Tumor). Regression is not suitable here because we are not predicting a continuous numerical value.

In [10]:
df['Intensity_Spread'] = df['Variance'] / (df['Mean'] + 1e-5)
df['Complexity_Score'] = df['Entropy'] * df['Homogeneity']

print("The two new features are: Intensity_Spread, Complexity_Score")

The two new features are: Intensity_Spread, Complexity_Score


Explanation of two new features:

The first one is Intensity_to_Variance_Ratio, calculated as Mean / (Variance + 1e-5). It basically shows how consistent the brightness of the image is. A higher value means the image is more uniform, while a lower value means the intensity varies a lot, which can be a sign of tumor regions.

The second feature is Texture_Complexity_Index, which is calculated by multiplying Entropy and Homogeneity. Entropy measures how random the image is, while Homogeneity shows how similar nearby pixels are. Combining them gives a better idea of the overall texture, helping distinguish between normal tissue and more irregular tumor areas.

In [11]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split 80/20 and scaled.")

Data split 80/20 and scaled.


In [12]:
# Algorithm 1: Logistic Regression
log_reg = LogisticRegression()
cv_scores_lr = cross_val_score(log_reg, X_train_scaled, y_train, cv=5)
log_reg.fit(X_train_scaled, y_train)

# Algorithm 2: Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores_rf = cross_val_score(rf_clf, X_train_scaled, y_train, cv=5)
rf_clf.fit(X_train_scaled, y_train)

print(f"LR 5-Fold CV Mean: {cv_scores_lr.mean():.4f}")
print(f"RF 5-Fold CV Mean: {cv_scores_rf.mean():.4f}")

LR 5-Fold CV Mean: 0.9824
RF 5-Fold CV Mean: 0.9865


In [13]:
def evaluate(model, X_t, y_t):
    preds = model.predict(X_t)
    return [accuracy_score(y_t, preds), precision_score(y_t, preds), 
            recall_score(y_t, preds), f1_score(y_t, preds)]

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Logistic Regression': evaluate(log_reg, X_test_scaled, y_test),
    'Random Forest': evaluate(rf_clf, X_test_scaled, y_test)
})

print(results)
print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_test, rf_clf.predict(X_test_scaled)))

      Metric  Logistic Regression  Random Forest
0   Accuracy             0.978378       0.990541
1  Precision             0.987539       0.993865
2     Recall             0.963526       0.984802
3   F1-Score             0.975385       0.989313

Confusion Matrix (Random Forest):
[[409   2]
 [  5 324]]


Conclusion

Between the two models, the Random Forest Classifier performed better than Logistic Regression, especially in terms of recall. This is important because in tumor detection, missing a tumor (false negative) is much worse than predicting one when it’s not there. The higher recall of Random Forest shows it is more reliable for this task.

In practical terms, the remaining errors (a few false negatives and false positives) mean that some tumor cases still look very similar to healthy tissue based on these features. Even though the accuracy is high, these mistakes are critical in a medical context. This suggests that while the model is strong, it should be used as a support tool rather than fully replacing medical experts, and more advanced methods could further improve the results.


In [14]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_clf, '../models/supervised_best.pkl')

X_test['actual'] = y_test
X_test['predicted'] = rf_clf.predict(X_test_scaled)
X_test['error'] = (X_test['actual'] != X_test['predicted']).astype(int)
X_test.to_csv('../data/residuals.csv', index=False)
